# PathoVision AI: training on Google Colab (free T4 GPU)

**Before running:** go to **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**.

- When it asks, allow **Google Drive** access. Models and results are saved to `MyDrive/pathovision/`, so a Colab disconnect does not lose finished work.
- If Colab disconnects, reconnect and **Run all** again. Finished steps are detected in Drive and skipped.
- Keep the tab open, and click inside the page every ~30-60 min (free Colab stops idle sessions).

| Step | What | Time (T4, approx.) |
|---|---|---|
| 1 | Setup + Drive | 2 min |
| 2 | Download PCam (~7.5 GB) | 5-15 min |
| 3 | Train ResNet-18 patch classifier | 25-45 min |
| 4 | Download 6 tumour + 2 normal CAMELYON16 slides | 10-20 min |
| 5 | Extract patches, train U-Net | 30-50 min |
| 6 | Whole-slide inference, evaluation, benchmark | 10 min |

## 1. Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then run again."
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = "/content/drive/MyDrive/pathovision"      # persistent: models, metrics, reports
DATA = "/content/data"                           # scratch: datasets (re-downloaded after a disconnect)
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
os.environ["DATA"] = DATA

In [ ]:
# Get the code: clone from GitHub, or upload pathovision-ai.zip if the repo isn't pushed yet
import subprocess
REPO_URL = "https://github.com/nirdoshk0228/pathovision-ai.git"
CODE = "/content/pathovision-ai"
if not os.path.exists(f"{CODE}/scripts"):            # missing, or an empty repo was cloned
    subprocess.run(["rm", "-rf", CODE])
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE])
    if not os.path.exists(f"{CODE}/scripts"):
        subprocess.run(["rm", "-rf", CODE])
        from google.colab import files
        print("Code not found on GitHub (repo empty or not pushed). Upload pathovision-ai.zip:")
        files.upload()
        !unzip -q -o pathovision-ai.zip -d /content
assert os.path.exists(f"{CODE}/scripts"), "code missing: upload pathovision-ai.zip"
%cd /content/pathovision-ai
!pip install -q openslide-python openslide-bin imagecodecs
!df -h /content | tail -1

## 2. PatchCamelyon (327,680 patches, 96x96 px @ 10x, slide-disjoint splits)

In [ ]:
CLS_DIR = f"{OUT}/runs/cls_resnet18"
if not os.path.exists(f"{CLS_DIR}/metrics.json"):
    !bash scripts/download_data.sh pcam
else:
    print("classifier already trained, skipping PCam download")

## 3. Patch classifier

ImageNet-pretrained ResNet-18, AdamW + one-cycle LR, mixed precision, HED stain + D4 augmentation.
The threshold is chosen on **validation**, and the final numbers are on the untouched **test** split with 8-view TTA.

In [ ]:
if not os.path.exists(f"{CLS_DIR}/metrics.json"):
    !python -m pathovision.training.train_classifier --data-format pcam_h5 --data-dir {DATA}/pcam \
        --arch resnet18 --epochs 5 --batch-size 256 --lr 1e-3 --workers 2 --tta 8 --out-dir {CLS_DIR}

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open(f"{CLS_DIR}/metrics.json"))["results"]
for split, r in m.items():
    print(f"{split:9s} AUC {r['roc_auc']:.4f}  acc {r['accuracy']:.4f}  precision {r['precision']:.4f}  recall {r['recall']:.4f}  F1 {r['f1']:.4f}")
display(Image(f"{CLS_DIR}/roc_test.png"))

In [ ]:
# free disk space for the slides (PCam no longer needed)
!rm -rf {DATA}/pcam

## 4. CAMELYON16 whole-slide images

From the AWS Open Data bucket `s3://camelyon-dataset` (no account needed). Each slide is 0.5-3 GB.

In [ ]:
!N_TUMOR=6 N_NORMAL=2 bash scripts/download_data.sh camelyon

## 5. Patch extraction + U-Net segmentation

Splits are **by slide**: one tumour slide for validation, and one tumour + one normal slide held out completely for step 6.

In [ ]:
import glob
imgs = sorted(glob.glob(f"{DATA}/camelyon16/images/*.tif")) + sorted(glob.glob(f"{DATA}/holdout/*.tif"))
names = sorted({os.path.basename(p)[:-4] for p in imgs})
tumour = [s for s in names if s.startswith("tumor")]
normal = [s for s in names if s.startswith("normal")]
HOLDOUT = [tumour[-1]] + ([normal[-1]] if normal else [])
VAL = tumour[-2]
os.makedirs(f"{DATA}/holdout", exist_ok=True)
for s in HOLDOUT:
    src = f"{DATA}/camelyon16/images/{s}.tif"
    if os.path.exists(src):
        os.replace(src, f"{DATA}/holdout/{s}.tif")
print("training slides:", [s for s in names if s not in HOLDOUT])
print("validation slide:", VAL, "| held out:", HOLDOUT)

In [ ]:
SEG_DIR = f"{OUT}/runs/seg_resnet34"
if not os.path.exists(f"{SEG_DIR}/metrics.json"):
    !python scripts/extract_patches.py --slides "{DATA}/camelyon16/images/*.tif" \
        --xml-dir {DATA}/camelyon16/annotations --out {DATA}/patches --val-slides {VAL} --test-slides none \
        --no-cls --seg-size 256 --seg-mpp 0.972 --seg-per-slide 600

In [ ]:
if not os.path.exists(f"{SEG_DIR}/metrics.json"):
    !python -m pathovision.training.train_segmentation --data-dir {DATA}/patches/seg \
        --arch resnet34_unet --epochs 20 --batch-size 16 --lr 3e-4 --workers 2 --tta 8 --out-dir {SEG_DIR}
print(json.load(open(f"{SEG_DIR}/metrics.json"))["results"])

## 6. Whole-slide inference on the held-out slides

In [ ]:
!mkdir -p {OUT}/models
!cp {CLS_DIR}/best.pt {OUT}/models/classifier.pt
!cp {SEG_DIR}/best.pt {OUT}/models/segmenter.pt
!python scripts/evaluate_slides.py --slides "{DATA}/holdout/*.tif" --xml-dir {DATA}/camelyon16/annotations \
    --cls-ckpt {OUT}/models/classifier.pt --seg-ckpt {OUT}/models/segmenter.pt \
    --out {OUT}/outputs/holdout --workers 2

In [ ]:
for d in sorted(glob.glob(f"{OUT}/outputs/holdout/*/")):
    r = json.load(open(d + "report.json"))
    q, b = r["quantification"], r["benchmark"]
    print(f"\n{os.path.basename(d[:-1])}: {r['slide']['width']:,} x {r['slide']['height']:,} px | "
          f"{b['tiles_classified']:,} patches ({b['tiles_skipped_background_pct']}% background skipped) | "
          f"{b['classification_tiles_per_sec']} patches/s | total {b['total_seconds']} s | "
          f"peak RAM {b['peak_rss_mb']} MB, GPU {b['peak_gpu_mb']} MB")
    print(f"  lesions {q['n_lesions']} | largest {q['largest_lesion_diameter_mm']:.2f} mm | {q['slide_category']} | "
          f"tumour {q['tumor_area_mm2']:.3f} mm2 | Dice {r.get('ground_truth', {}).get('dice')}")
    display(Image(d + "heatmap_overlay.png", width=700))
    display(Image(d + "prediction_overlay.png", width=700))
print(open(f"{OUT}/outputs/holdout/summary.json").read())

In [ ]:
# Throughput vs batch size (for the README performance table)
bench_slide = sorted(glob.glob(f"{DATA}/holdout/tumor_*.tif"))[0]
!python scripts/benchmark.py --slide {bench_slide} --cls-ckpt {OUT}/models/classifier.pt --batch-sizes 64 256 512 --workers 2

## Done

Everything is in Google Drive → `MyDrive/pathovision/`:
- `models/classifier.pt`, `models/segmenter.pt`
- `runs/*/metrics.json`, `runs/cls_resnet18/roc_test.png`: the numbers for README and resume
- `outputs/holdout/<slide>/heatmap_overlay.png` and `prediction_overlay.png`: screenshots for `assets/`